# 🤖 AI-Powered Notebook Updater

This notebook uses the **Gemini API** to intelligently update cells in target notebooks based on natural language instructions. It's context-aware, safe, and easy to use.

---

## 📋 Prerequisites

Before running this notebook, you'll need:

1. **Gemini API Key** from your organization's GenAI platform
2. **Databricks Personal Access Token** (generate from User Settings → Access Tokens)
3. **Target notebook** with cells marked for update (see format below)
4. **Active compute cluster** attached to this notebook

---

## 🚀 Quick Start Guide

Follow these steps in order:

### Step 1: Configure Credentials
1. **Run Cell 2** to create input widgets at the top of this notebook
2. **Fill in the widgets** that appear:
   - `Gemini API Key`: Your API key (required)
   - `Base URL`: Pre-filled with `https://api.genai.mil/v1` (verify or update)
   - `Databricks Token`: Your personal access token (required)
   - `Username`: Your email address (optional)

### Step 2: Initialize Functions
3. **Run Cell 3** to load all functions and connect to the Gemini API
   - First run will install `databricks-sdk` (requires re-run after install)
   - Watch for "✅ ALL FUNCTIONS LOADED AND READY" confirmation

### Step 3: Prepare Your Target Notebook
4. In your target notebook, mark cells for AI update with this format:
   ```python
   # update this cell
   # Add detailed instructions here about what changes you want
   # For example: "Transform this to use PySpark instead of pandas"
   # or "Add error handling and logging to this API call"
   
   # Your existing code here...
   ```

### Step 4: Run the Update
5. **Edit Cell 4** to set your target notebook path:
   ```python
   TARGET_NOTEBOOK_PATH = "/Workspace/your_folder/YourNotebook"
   ```
6. **Run Cell 4** with `DRY_RUN = True` first to preview changes
7. If satisfied, set `DRY_RUN = False` and run again to apply

### Step 5: Revert if Needed (Optional)
8. If you need to undo changes, **uncomment and run Cell 5**
   - Restores from the automatic backup created in Step 4

---

## 🎯 How It Works

* **Context-Aware**: Reads your entire notebook to understand relationships between cells
* **Selective**: Only modifies cells with the `# update this cell` marker
* **Safe**: Creates automatic backups before making changes
* **Instruction-Driven**: Uses your markdown comments to guide AI transformations

---

## 💡 Tips

* Be specific in your update instructions for best results
* The AI can see all cells, so reference other parts of your notebook in instructions
* Start with `DRY_RUN = True` to preview before applying changes
* Lower temperature (0.2-0.4) gives more focused results; higher (0.6-0.8) more creative

In [0]:
# ==========================================
# CREATE CREDENTIAL INPUT WIDGETS
# ==========================================
# Notebook widgets provide a secure way to input credentials

dbutils.widgets.removeAll()

# Create widgets for Gemini API credentials
dbutils.widgets.text("api_key", "", "Gemini API Key")
dbutils.widgets.text("base_url", "https://api.genai.mil/v1", "Base URL")
dbutils.widgets.text("databricks_token", "", "Databricks Token")
dbutils.widgets.text("username", "", "Username (optional)")

print("✅ Input widgets created successfully!")
print("")
print("🔑 NEXT STEP: Fill in the widgets at the top of this notebook:")
print("   1. Gemini API Key: Enter your API key (e.g., 'sk-...')")
print("   2. Base URL: Verify or update (default: https://api.genai.mil/v1)")
print("   3. Databricks Token: Enter your Personal Access Token (generate from User Settings → Access Tokens)")
print("   4. Username (optional): Your email address for tracking")
print("")
print("⚠️  SECURITY NOTE: These credentials are stored only in widget memory (not saved in notebook)")

✅ Input widgets created successfully!

🔑 NEXT STEP: Fill in the widgets at the top of this notebook:
   1. Gemini API Key: Enter your API key (e.g., 'sk-...')
   2. Base URL: Verify or update (default: https://api.genai.mil/v1)
   3. Databricks Token: Enter your Personal Access Token (generate from User Settings → Access Tokens)
   4. Username (optional): Your email address for tracking

⚠️  SECURITY NOTE: These credentials are stored only in widget memory (not saved in notebook)


In [0]:
# ==========================================
# INITIALIZE: ALL FUNCTIONS AND API CONNECTION
# ==========================================

# Install Databricks SDK if needed
# Note: This will restart the Python interpreter, clearing all variables.
# After running this cell once, you can comment out the pip install line.
try:
    from databricks.sdk import WorkspaceClient
    sdk_installed = True
except ImportError:
    sdk_installed = False

if not sdk_installed:
    %pip install -q databricks-sdk
    # The interpreter will restart here. Re-run this cell after restart.
    raise SystemExit("SDK installed. Please re-run this cell to load all functions.")

import json
import requests
import re
import time
import base64
from typing import Dict, List, Any

# Read credentials from widgets
api_key = dbutils.widgets.get("api_key")
base_url = dbutils.widgets.get("base_url")
databricks_token = dbutils.widgets.get("databricks_token")
username = dbutils.widgets.get("username")

if not api_key:
    raise ValueError("❌ Gemini API Key is required. Please enter your API key in the widget at the top.")

if not databricks_token:
    raise ValueError("❌ Databricks Token is required. Please enter your token in the widget at the top.")

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

print("✅ Credentials loaded")

# ==========================================
# SELECT MODEL AND TEST SELECTION
# ==========================================

print("🔍 Connecting to Gemini API...")
try:
    response = requests.get(f"{base_url}/models", headers=headers)
    response.raise_for_status()
    models_data = response.json()
    available_models = [model.get('id', 'unknown') for model in models_data.get('data', [])]
    SELECTED_MODEL = 'gemini-3.7-flash' # <------ Paste selected model here
    print(f"✅ Connected! Using model: {SELECTED_MODEL}")
except Exception as e:
    raise RuntimeError(f"❌ Failed to connect: {e}")

# ==========================================
# CORE FUNCTIONS
# ==========================================

def identify_update_cells(cells: List[Dict]) -> List[Dict]:
    """Find cells marked with '# update this cell' and extract instructions."""
    update_cells = []
    for idx, cell in enumerate(cells):
        # Handle JUPYTER format (source is a list) and Databricks format (content is a string)
        if 'source' in cell:
            content = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        else:
            content = cell.get('content', '')
        if '# update this cell' in content.lower():
            lines = content.split('\n')
            instructions = []
            in_instructions = False
            for line in lines:
                if '# update this cell' in line.lower():
                    in_instructions = True
                    continue
                if in_instructions and line.strip().startswith('#'):
                    instructions.append(line.strip().lstrip('#').strip())
                elif in_instructions and line.strip() and not line.strip().startswith('#'):
                    break
            update_cells.append({
                'index': idx,
                'cell_id': cell.get('cellId', cell.get('cell_guid', '')),
                'nuid': cell.get('nuid', cell.get('cell_nuid', '')),
                'original_content': content,
                'instructions': ' '.join(instructions) if instructions else 'Update this cell based on context',
                'language': cell.get('cell_type', cell.get('language', 'python'))
            })
    return update_cells

def generate_context_summary(cells: List[Dict], max_cells: int = 50) -> str:
    """Generate full notebook context for the AI."""
    summary = "NOTEBOOK CONTEXT:\n\n"
    context_cells = cells[:max_cells] if len(cells) > max_cells else cells
    for idx, cell in enumerate(context_cells):
        # Handle JUPYTER format (source is a list) and Databricks format (content is a string)
        if 'source' in cell:
            content = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        else:
            content = cell.get('content', '')
        if len(content) > 500:
            content = content[:500] + "...\n[Content truncated]"
        summary += f"--- Cell {idx + 1}: ({cell.get('cell_type', cell.get('language', 'unknown'))}) ---\n{content}\n\n"
    return summary

def call_gemini_for_update(context: str, cell_to_update: Dict, temperature: float = 0.3) -> str:
    """Call Gemini API to generate updated cell content."""
    
    # Simpler, more direct prompt
    prompt = f"""TASK: Generate PySpark code for Databricks based on these instructions.

INSTRUCTIONS: {cell_to_update['instructions']}

CONTEXT FROM NOTEBOOK:
{context}

ORIGINAL CODE:
{cell_to_update['original_content']}

REQUIREMENTS:
1. Use PySpark (spark.createDataFrame, pyspark.sql.functions) for data operations
2. Each Python statement MUST be on its own line with a newline character after it
3. Add blank lines between logical sections
4. Remove '# update this cell' marker and instruction comments
5. Add a brief 2-3 line comment at the TOP explaining what was changed and what to do next
6. Include necessary imports
7. Return ONLY executable code, no markdown fences, no explanations

EXAMPLE OUTPUT FORMAT:
# AI Updated: Created PySpark DataFrame with employee data and filtered by salary
# Next: Run this cell to see employees earning over $50k

from pyspark.sql import functions as F

data = [...]
...

GENERATE CODE NOW:
"""
    
    payload = {
        "model": SELECTED_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": 4000
    }
    response = requests.post(f"{base_url}/chat/completions", headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    updated_content = response.json()['choices'][0]['message']['content'].strip()
    
    # Clean up any markdown fences
    updated_content = re.sub(r'^```[a-zA-Z]*\n', '', updated_content)
    updated_content = re.sub(r'\n```$', '', updated_content)
    updated_content = updated_content.strip()
    
    # POST-PROCESSING: Fix concatenated code with multi-pass regex
    # Apply ALWAYS, not just as fallback
    # Apply multiple times to catch all cases
    for _ in range(3):  # Multiple passes to catch chained statements
        # Add newline after closing bracket ] when followed by identifier (but not inside brackets)
        updated_content = re.sub(r'(\])([a-zA-Z_])', r'\1\n\n\2', updated_content)
        # Add newline after closing paren ) when followed by identifier starting new statement
        updated_content = re.sub(r'(\))([a-zA-Z_][a-zA-Z0-9_]*\s*=)', r'\1\n\n\2', updated_content)
        # Add newline before display/print/show calls
        updated_content = re.sub(r'([a-zA-Z0-9_\)\]])(display\(|print\(|show\()', r'\1\n\n\2', updated_content)
        # Add newline after complete import lines (greedy to capture full line)
        updated_content = re.sub(r'(^import [^\n]+$)([a-zA-Z_])', r'\1\n\n\2', updated_content, flags=re.MULTILINE)
        updated_content = re.sub(r'(^from [^\n]+ import [^\n]+$)([a-zA-Z_])', r'\1\n\n\2', updated_content, flags=re.MULTILINE)
    
    # Clean up excessive newlines
    updated_content = re.sub(r'\n{3,}', '\n\n', updated_content)
    
    # Validate syntax for Python code
    if cell_to_update['language'].lower() in ['python', 'py', 'code']:
        try:
            compile(updated_content, '<string>', 'exec')
        except SyntaxError as e:
            raise ValueError(f"Generated code has syntax errors: {e}\n\nCode:\n{updated_content}")
    
    return updated_content

def process_notebook_updates(notebook_path: str, temperature: float = 0.3, dry_run: bool = True):
    """Main function: read notebook, generate updates, apply changes."""
    global BACKUP_PATH, LAST_UPDATE_RESULTS
    
    # Import SDK only when needed
    try:
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service import workspace as ws_service
        # Use the Databricks token from widgets for authentication
        w = WorkspaceClient(token=databricks_token)
    except ImportError:
        raise RuntimeError("❌ Databricks SDK not available. Install with: %pip install databricks-sdk")
    
    print("="*70)
    print("🚀 NOTEBOOK UPDATE PROCESS")
    print("="*70)
    
    # Read notebook
    print(f"\n📖 Reading: {notebook_path}")
    notebook_export = w.workspace.export(notebook_path, format=ws_service.ExportFormat.JUPYTER)
    notebook_json_str = base64.b64decode(notebook_export.content).decode('utf-8')
    notebook_data = json.loads(notebook_json_str)
    cells = notebook_data.get('cells', [])
    print(f"   ✅ Found {len(cells)} cells")
    
    # Create backup (only if not dry run)
    BACKUP_PATH = None
    if not dry_run:
        print(f"\n💾 Creating backup...")
        # Use fixed backup name (overwrites previous backup)
        BACKUP_PATH = f"{notebook_path}_BACKUP"
        # Export and re-import to create backup copy
        backup_export = w.workspace.export(notebook_path, format=ws_service.ExportFormat.JUPYTER)
        w.workspace.import_(BACKUP_PATH, format=ws_service.ImportFormat.JUPYTER, 
                           content=backup_export.content, overwrite=True)
        print(f"   ✅ Backup saved (overwrites previous): {BACKUP_PATH}")
    
    # Find cells to update
    print(f"\n🔍 Finding cells to update...")
    update_cells = identify_update_cells(cells)
    if not update_cells:
        print("   ⚠️  No cells found with '# update this cell' marker")
        return {'success': False, 'message': 'No cells marked for update'}
    print(f"   ✅ Found {len(update_cells)} cell(s)")
    for cell in update_cells:
        print(f"      • Cell {cell['index'] + 1}: {cell['instructions'][:60]}...")
    
    # Generate context
    print(f"\n🌐 Generating context from all cells...")
    context = generate_context_summary(cells)
    
    # Generate updates
    print(f"\n🤖 Generating updates with Gemini...")
    updates = []
    for i, cell in enumerate(update_cells):
        print(f"\n   [{i+1}/{len(update_cells)}] Cell {cell['index'] + 1}: {cell['instructions'][:50]}...")
        try:
            updated_content = call_gemini_for_update(context, cell, temperature)
            updates.append({
                'cell_index': cell['index'],
                'original': cell['original_content'],
                'updated': updated_content,
                'instructions': cell['instructions']
            })
            print(f"       ✅ Generated ({len(updated_content)} chars)")
        except Exception as e:
            print(f"       ❌ Error: {e}")
            updates.append({'cell_index': cell['index'], 'error': str(e)})
    
    # Apply or preview updates
    if dry_run:
        print(f"\n👁️  DRY RUN - Preview of updates:")
        for update in updates:
            if 'error' not in update:
                print(f"\n   Cell {update['cell_index'] + 1}:")
                print(f"   - Original: {len(update['original'])} chars")
                print(f"   - Updated: {len(update['updated'])} chars")
                print(f"   - First 100 chars: {update['updated'][:100]}...")
    else:
        print(f"\n✍️  Applying updates...")
        for update in updates:
            if 'error' not in update:
                # Split by newlines and preserve \n at end of each line (Jupyter format)
                lines = update['updated'].split('\n')
                # Add \n back to all lines except handle empty last line
                cells[update['cell_index']]['source'] = [line + '\n' for line in lines[:-1]] + ([lines[-1]] if lines[-1] else [])
        updated_json = json.dumps(notebook_data, indent=2)
        w.workspace.import_(notebook_path, format=ws_service.ImportFormat.JUPYTER, 
                           content=base64.b64encode(updated_json.encode('utf-8')).decode('utf-8'), 
                           overwrite=True)
        print(f"   ✅ Updated {len([u for u in updates if 'error' not in u])} cell(s)")
    
    # Store results
    LAST_UPDATE_RESULTS = {
        'notebook_path': notebook_path,
        'backup_path': BACKUP_PATH,
        'updates': updates,
        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S"),
        'dry_run': dry_run
    }
    
    print("\n" + "="*70)
    print("✅ COMPLETE")
    print("="*70)
    return {'success': True, 'cells_updated': len([u for u in updates if 'error' not in u]), 'backup_path': BACKUP_PATH}

def revert_to_backup():
    """Restore notebook from most recent backup."""
    if not LAST_UPDATE_RESULTS or LAST_UPDATE_RESULTS.get('dry_run') or not LAST_UPDATE_RESULTS.get('backup_path'):
        print("❌ No backup available to revert")
        return False
    
    try:
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service import workspace as ws_service
        # Use the Databricks token from widgets for authentication
        w = WorkspaceClient(token=databricks_token)
    except ImportError:
        raise RuntimeError("❌ Databricks SDK not available. Install with: %pip install databricks-sdk")
    notebook_path = LAST_UPDATE_RESULTS['notebook_path']
    backup_path = LAST_UPDATE_RESULTS['backup_path']
    print(f"↩️  Reverting {notebook_path}...")
    # Export backup and re-import to original location
    backup_export = w.workspace.export(backup_path, format=ws_service.ExportFormat.JUPYTER)
    w.workspace.import_(notebook_path, format=ws_service.ImportFormat.JUPYTER, 
                       content=backup_export.content, overwrite=True)
    print(f"✅ Reverted! Backup still at: {backup_path}")
    return True

# Initialize globals
BACKUP_PATH = None
LAST_UPDATE_RESULTS = None

print("\n" + "="*70)
print("✅ ALL FUNCTIONS LOADED AND READY")
print("="*70)

✅ Credentials loaded
🔍 Connecting to Gemini API...
✅ Connected! Using model: gemini-3.7-flash

✅ ALL FUNCTIONS LOADED AND READY


In [0]:
# ==========================================
# SET TARGET NOTEBOOK PATH AND RUN UPDATE
# ==========================================
# NOTE: You must run Cell 3 first to load all functions!

# Configure your target notebook
TARGET_NOTEBOOK_PATH = "/Workspace/your_folder/YourNotebookName"  # REPLACE with your notebook path
UPDATE_TEMPERATURE = 0.3  # 0.0 (focused) to 1.0 (creative) - 0.3 recommended for better formatting
DRY_RUN = False  # Set to True to preview, False to apply changes

# Run the update
results = process_notebook_updates(
    notebook_path=TARGET_NOTEBOOK_PATH,
    temperature=UPDATE_TEMPERATURE,
    dry_run=DRY_RUN
)

if results['success']:
    if results.get('cells_updated', 0) > 0:
        if DRY_RUN:
            print(f"\n💡 TIP: Review the preview above. If satisfied, set DRY_RUN=False and run again.")
        else:
            print(f"\n✅ Changes applied! Backup saved at: {results.get('backup_path')}")
else:
    print(f"\n❌ {results.get('message', 'Update failed')}")

🚀 NOTEBOOK UPDATE PROCESS

📖 Reading: /Workspace/jup-swctc-r/SMWDC/Dev/Ryan/Agentic Notebook Test
   ✅ Found 2 cells

💾 Creating backup...
   ✅ Backup saved (overwrites previous): /Workspace/jup-swctc-r/SMWDC/Dev/Ryan/Agentic Notebook Test_BACKUP

🔍 Finding cells to update...
   ✅ Found 1 cell(s)
      • Cell 1: my eyes hurt make it light mode AI Updated: Created a PySpar...

🌐 Generating context from all cells...

🤖 Generating updates with Gemini...

   [1/1] Cell 1: my eyes hurt make it light mode AI Updated: Create...
       ✅ Generated (8853 chars)

✍️  Applying updates...
   ✅ Updated 1 cell(s)

✅ COMPLETE

✅ Changes applied! Backup saved at: /Workspace/jup-swctc-r/SMWDC/Dev/Ryan/Agentic Notebook Test_BACKUP


In [0]:
# ==========================================
# REVERT TO BACKUP
# ==========================================

# Uncomment the line below to revert the most recent update
# revert_to_backup()

print("⚠️  To revert: uncomment revert_to_backup() above and run this cell")

---

# 📝 Example: Target Notebook Template

This shows how to structure cells in your **target notebook** for AI updates.

## Example Cell to Update

```python
# update this cell
# Transform the data to calculate 7-day rolling averages for the 'sales' column
# grouped by 'region'. Add a new column called 'sales_7day_avg'.

import pandas as pd

# Current basic code
df = spark.table("sales_data").toPandas()
print(df.head())
```

## Example Cell (Context Only - No Updates)

```python
# This cell won't be updated because it doesn't have the marker

import pyspark.sql.functions as F
from pyspark.sql import Window

# Load data
df = spark.table("main.analytics.sales_data")
```

## Another Example Cell to Update

```python
# update this cell
# Add comprehensive error handling, retry logic, and logging to this API call.
# Use exponential backoff for retries (max 3 attempts).

import requests

response = requests.get("https://api.example.com/data")
data = response.json()
```

---

**Key Points:**
* Only cells with `# update this cell` are modified
* The AI reads ALL cells for context
* Be specific in your instructions
* Use markdown-style comments (#) for instructions